## 앙상블학습_스태킹회귀

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('house_prices.csv')
df

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,0.06263,0.0,11.93,0,0.573,6.593,69.1,2.4786,1,273,21.0,391.99,9.67,22.4
502,0.04527,0.0,11.93,0,0.573,6.120,76.7,2.2875,1,273,21.0,396.90,9.08,20.6
503,0.06076,0.0,11.93,0,0.573,6.976,91.0,2.1675,1,273,21.0,396.90,5.64,23.9
504,0.10959,0.0,11.93,0,0.573,6.794,89.3,2.3889,1,273,21.0,393.45,6.48,22.0


In [3]:
features = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX',
       'PTRATIO', 'B', 'LSTAT']

features = ['CRIM', 'ZN', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX',
       'PTRATIO', 'B', 'LSTAT']

X = df[features]
y = df['MEDV']

In [4]:
# 트레이닝/테스트 데이터 분할
from sklearn.model_selection import train_test_split
X_tn, X_te, y_tn, y_te = train_test_split(X, y, random_state=0)

In [5]:
# 트레이닝/테스트 데이터 분할
from sklearn.model_selection import train_test_split
X_tn, X_te, y_tn, y_te = train_test_split(X, y, random_state=0)

In [6]:
# 데이터 표준화
from sklearn.preprocessing import StandardScaler
std_scale = StandardScaler()
std_scale.fit(X_tn)
X_tn_std = std_scale.transform(X_tn)
X_te_std = std_scale.transform(X_te)

In [7]:
# 스태킹 학습

from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import StackingRegressor

clf1 = SVR(kernel='linear')
clf2 = DecisionTreeRegressor(random_state=0)

clf_stkg = StackingRegressor(
    estimators=[
    ('svm', clf1),
    ('tree', clf2)],

    final_estimator=LinearRegression())

clf_stkg.fit(X_tn_std, y_tn)

StackingRegressor(estimators=[('svm', SVR(kernel='linear')),
                              ('tree', DecisionTreeRegressor(random_state=0))],
                  final_estimator=LinearRegression())

In [8]:
# 예측
stkg_pred = clf_stkg.predict(X_te_std)
print(stkg_pred)

[25.07312933 23.46137633 24.78893691 14.24329904 21.2179586  20.19558945
 20.64635808 19.88429451 15.88628703 17.16135536  7.56800984 17.07026299
 15.33617211  6.98856975 46.53421088 35.94265197 21.40574169 35.74233093
 25.44533738 20.69139687 24.02373951 19.09409858 19.2449907  26.56900815
 21.05688313 11.70516432 15.7274693  17.92489745 39.07805983 19.44224284
 13.90116373 18.96212962 23.10694253 20.95748535 24.53646652 20.50192662
 10.5939852  22.15356312 13.44042468 17.01393142 25.29088752 23.43152433
 23.1248722  12.17044503 22.82290661 21.04210564 21.10729029 16.77231892
 12.78015978 27.59576897 15.64367104 19.48201211 23.78681182 26.46563012
 13.59757929 20.02002031 20.19760756 17.2830549   9.9902121  21.54860111
 20.21945018 22.86004715 30.95929261 25.51611935 21.82874553 26.98836261
 18.88709369 19.59706127 13.13003062 23.19262953 21.41871756 23.54541297
 26.13463304 30.10928443 26.04787183  6.34517777 41.75112919 23.57463537
 23.94295439 22.8271778  27.67569282 18.60055106 19

In [9]:
# 모형 평가-MSE
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_te, stkg_pred)
print(mse)

26.645162459957184
